In [1]:
import pandas

/Users/karthickkumarasamy/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df =pd.read_csv('Detailed_Polling_Data.csv')

<IPython.core.display.Javascript object>

In [3]:
df.columns

Index(['Serial No. Of Polling Station', 'Locality',
       'Building  in  Which  it  will  be  Located', 'Polling  Area',
       'Whether  for  all Voters  or  men  olny or  women  only',
       'Indian National Congress', 'Naam Tamilar Katchi',
       'All India Anna Dravida Munnetra Kazhagam', 'Tamilaga Vettri Kazhagam',
       'Nam Naadu Nam Makkal Nam Ethirkaalam Katchi',
       'Puthiya Makkal Tamil Desam Katchi', 'Naam Indiar Party',
       'Vishwa Tamil Kazhagam', 'All India Forward Bloc', 'Puthiya Tamilagam',
       'Tamizhaga Vaazhvurimai Katchi', 'Independent', 'Independent.1',
       'Independent.2', 'Independent.3', 'Total of Valid Votes',
       'No. Of Rejected Votes', 'NOTA', 'Total', 'No. Of Tendered Votes',
       'Indian National Congress_Share_%', 'Naam Tamilar Katchi_Share_%',
       'All India Anna Dravida Munnetra Kazhagam_Share_%',
       'Tamilaga Vettri Kazhagam_Share_%',
       'Nam Naadu Nam Makkal Nam Ethirkaalam Katchi_Share_%',
       'Puthiya Makkal Tamil

In [5]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Load the new dataset (Update the filename to match your file)
df5 = pd.read_csv("Detailed_Polling_Data.csv")

# 2. Select the key political party share columns present in this index
new_party_shares = [
    'Indian National Congress_Share_%',
    'Tamilaga Vettri Kazhagam_Share_%',
    'All India Anna Dravida Munnetra Kazhagam_Share_%',
    'Naam Tamilar Katchi_Share_%',
    'Puthiya Tamilagam_Share_%',
    'Tamizhaga Vaazhvurimai Katchi_Share_%'
]

# Handle any missing data in the share columns by making them 0
df5[new_party_shares] = df5[new_party_shares].fillna(0)

# 3. Calculate Independent Share % explicitly for feature depth
# (Using your pre-calculated columns)
df5['Total_Independent_Votes'] = df5['Total_Independent_Votes'].fillna(0)
df5['Total of Valid Votes'] = df5['Total of Valid Votes'].replace(0, np.nan) # Avoid division by zero
df5['Independent_Share_%'] = (df5['Total_Independent_Votes'] / df5['Total of Valid Votes']) * 100

# 4. Include structural features for the voter behavior analysis
feature_cols = new_party_shares + ['Independent_Share_%', 'Margin_Percentage']
df5[feature_cols] = df5[feature_cols].fillna(0)

# 5. Extract and scale the features
X = df5[feature_cols]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. Apply K-Means Clustering to group booths into 4 core segments
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df5['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# 7. Print the Profile Breakdown of each voter group
print("\n--- DATASET 5: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---")
profile = df5.groupby('Cluster_ID')[feature_cols].mean()
print(profile.round(2))

print("\n--- DATASET 5: BOOTH COUNT PER CLUSTER ---")
print(df5['Cluster_ID'].value_counts())

# 8. Automatically export separate action lists for campaign ground teams
for cluster_num in range(optimal_k):
    cluster_df = df5[df5['Cluster_ID'] == cluster_num][
        [
            'Serial No. Of Polling Station', 
            'Locality', 
            'Building  in  Which  it  will  be  Located', 
            'Polling  Area', 
            'Winner_Party', 
            'Margin_Percentage'
        ]
    ]
    filename = f"Dataset_5_Cluster_{cluster_num}_Booths.csv"
    cluster_df.to_csv(filename, index=False)

print("\nSuccess! Campaign target files generated for all 4 clusters.")



--- DATASET 5: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---
            Indian National Congress_Share_%  \
Cluster_ID                                     
0                                      60.87   
1                                      23.83   
2                                      15.84   
3                                      18.83   

            Tamilaga Vettri Kazhagam_Share_%  \
Cluster_ID                                     
0                                      20.15   
1                                      37.49   
2                                      27.98   
3                                      35.72   

            All India Anna Dravida Munnetra Kazhagam_Share_%  \
Cluster_ID                                                     
0                                                      12.54   
1                                                      28.81   
2                                                      45.16   
3                                        